# 03 — Financial Linkage Analysis
**PCA Macro Slowdown Indicator**
Siddhanth Yadav · Kavin Dhanasekar · Sudhan Adithya

This notebook covers:
- Contemporaneous correlations (PC1 vs. financial variables)
- Simple OLS regressions (financial ~ PC1)
- Lead-lag regressions: do financial variables anticipate PC1?
- Lead-lag β coefficient chart

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
load_dotenv('../.env')

# Load processed panel (run main.py first, or notebook 02)
import config
from analysis.financial_linkage import compute_correlations, contemporaneous_regressions, regression_summary_table
from analysis.lead_lag import bivariate_lead_lag, lead_lag_regression
from viz.charts import plot_correlation_heatmap, plot_lead_lag_betas

# Load outputs produced by main.py
pc1       = pd.read_csv('../outputs/pc1_series.csv', index_col=0, parse_dates=True).squeeze()
panel_std = pd.read_csv('../data/processed/panel_clean.csv', index_col=0, parse_dates=True)

fin_cols  = [c for c in config.FINANCIAL_COLS if c in panel_std.columns]
fin_panel = panel_std[fin_cols]
print('Financial columns:', fin_cols)

In [ ]:
# --- Contemporaneous correlations ---
corr_table = compute_correlations(pc1, fin_panel)
print('Contemporaneous Pearson Correlations:')
corr_table

In [ ]:
# --- Correlation heatmap ---
combined    = pd.concat([pc1.rename('pc1'), fin_panel], axis=1).dropna()
corr_matrix = combined.corr()
fig, ax = plot_correlation_heatmap(corr_matrix)
plt.show()

In [ ]:
# --- Contemporaneous OLS regressions (financial ~ PC1) ---
reg_results = contemporaneous_regressions(pc1, fin_panel, direction='financial_on_pc1')
reg_table   = regression_summary_table(reg_results)
print('\nOLS Summary (financial ~ PC1):')
reg_table

In [ ]:
# --- Lead-lag bivariate regressions ---
ll_df = bivariate_lead_lag(pc1, fin_panel, leads=[1, 3, 6])
print('\nLead-Lag Results (PC1_{t+h} ~ financial_t):')
ll_df

In [ ]:
# --- Lead-lag β chart ---
fig, ax = plot_lead_lag_betas(ll_df)
plt.show()

In [ ]:
# --- Joint multivariate lead-lag (h=1) ---
joint_results = lead_lag_regression(pc1, fin_panel, leads=[1, 3, 6])
for h, res in joint_results.items():
    print(f'\n=== h = {h} months ===')
    print(res.summary2())